# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant metadata JSON-LD)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview

Let's list available record sets, their `@id`s, and their fields.

In [ ]:
# Discover available record sets and their schema
record_sets = []
print("Available data record sets and fields:")
for rs in metadata.record_sets:
    print(f"- Record set name: {getattr(rs, 'name', None)}  (@id={rs.id})")
    record_sets.append(rs.id)
    # List fields for each record set
    for field in rs.fields:
        print(f"    - Field: {getattr(field, 'name', None)}  (@id={field.id})  [dataType={getattr(field, 'data_type', None)}]")
print("\nTotal record sets found:", len(record_sets))

## 3. Data Extraction

Load data from the record sets identified above into pandas DataFrames. We will refer to all record sets and field names by their `@id` values.


In [ ]:
# Load records from all record sets into DataFrames
dataframes = dict()
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records for record set {rs_id}.")
    dataframes[rs_id] = df

# For demonstration, use the first available record set
if record_sets:
    record_set_id = record_sets[0]
    print(f"\nColumns in record set '@id': {record_set_id}")
    print(list(dataframes[record_set_id].columns))
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply basic analysis steps: filtering records by a numeric field, normalizing, grouping, or summarizing data as appropriate.


In [ ]:
# Find the first numeric field in this record set for demonstration
record_set_obj = None
for rs in metadata.record_sets:
    if rs.id == record_set_id:
        record_set_obj = rs
        break

numeric_field_id = None
group_field_id = None
for field in record_set_obj.fields:
    # Common Croissant numeric types
    if field.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number'] and numeric_field_id is None:
        numeric_field_id = field.id
    if (field.data_type in ['Text', 'schema:Text'] or 'name' in str(field.id).lower()) and group_field_id is None:
        group_field_id = field.id
if numeric_field_id is None:
    print("No numeric field found for analysis.")
else:
    print(f\
        f"Using numeric field (@id): {numeric_field_id}" + \
        (f" and group field (@id): {group_field_id}" if group_field_id else "")
    )
    # Handle missing or non-numeric values if present
    df = dataframes[record_set_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() is not None else 0
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical/text field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization

Visualize distributions and relationships between fields in the selected record set. We'll use histograms and scatterplots if suitable fields are available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
if numeric_field_id and numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if group_field_id and numeric_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to load, inspect, and analyze the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset defined in Croissant format. We explored record sets, dynamically referenced all entities by their `@id`, performed simple filtering and normalization of numeric fields, and visualized data distributions. Further domain-specific analysis is encouraged using the full field and record structure!